# 05 train model
- Research candidates and evaluate approaches
- Decision matrix to shortlist candidates
- Implement baseline and challenger model with default/ basic settings
- Evaluate both models
- Hyperparameter tuning
- Production model selection

# Research:

## Candidate #1
### TF-IDF + Logistic Regression
##### TF-IDF:
Feature extraction technique used in NLP to transform raw text into a numeric value that ML models can process/ understand.
The numerical value is a weight to each word based on two factors:
1. How often a word appears in a document/ record (TF, Term frequency)
2. How rare the word is across all documents (IDF, Inverse document frequency)

This helps the model to focus on words that actually inform it on differences between records by stripping away common unhelpful words and highlighting distinctive words in each record so the model can learn differences.

Reference: https://scikit-learn.org/1.4/tutorial/text_analytics/working_with_text_data.html

##### Logistic regression:
Linear classification algorithm used for both binary and multi-class classification problems. Since our problem is multi-class (i.e. we have more than two possible outcomes/ labels) we can use scikit-learn multinominal logistic regression to estimate the probability of each possible class using a softmax function.

Using the TF-IDF transformed data, the model can then learn a weight for each feature (a word) and an associated class. We then select the class with the highest probability as the prediction.

During inference the model calculates the probability for each possible resolver group and the class with the highest probability is selected as the final predicted value.

Reference: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

##### Overview of combination
TF-IDF + Logistic regression will be our baseline as it is computationally efficient, simple to explain and interpret and produces a probabalistic output to our multi-class classification problem.

Pros:
- Simple... basically text input, TF-IDF, Logistic regression = predicted resolver group. Low number of steps makes it easy to explain and implement
- Quick to train when compared with more complex neural network models so we can quickly experiment and iterate
- No black box processing, we can actually view the feature weights and see what words appear to be influencing classification decisions
- Probabilistic output allows us to route incidents for manual inspection if the confidence is low

Cons:
- TF-IDF just captures individual words, losing the meaning and the context of the words in the record
- Becuase each word is treated as a feature, the feature matrices can grow fairly large on a large number of records with lots of distinct words

Summary:
TF-IDF and logistic regression provide a good baseline which we can interpret and inspect as humans to understand it's behaviour. Where we may run into issues is the loss of context and meaning since TF-IDF treats individual words as features instead of phrases etc, we may find the model struggles with less obvious text descriptions or a large amount of variance in the words used for incidents of the same class.

To overcome these cons, other candidates will consider more advanced transformer based models.

____________________________________________________________________________________________________________________________________________

## Candidate #2
### DistilBERT - Transformer based language model
##### Transformer element
Transformers are a type of neural network design for processing natural language. The main difference between transformers and traditional NLP techniques like TF-IDF is that instead of treating individual words independently, transformers actually analyse the relationship between words in a sentence/ body of text using a mechanism called self-attention. This enables the model to understand not just words but the actual meaning of words in context.

References: 
- https://huggingface.co/docs/transformers/model_doc/distilbert
- https://www.ibm.com/think/topics/self-attention

##### DistilBERT
DistilBERT is a smaller and faster version of the BERT transformer model. It was created using a technique called knowledge distillation, which is where a smaller model is trained to replicate the behaviour of a larger pre-trained model. The result of that is a model that maintains a lot of the predictive power of the parent model but uses significantly less parameters and provides faster inference. This makes DistilBERT a good option when considering trade offs between performance/ computational cost.

References: 
- https://huggingface.co/docs/transformers/model_doc/bert
- https://huggingface.co/docs/transformers/model_doc/distilbert
- https://towardsdatascience.com/everything-you-need-to-know-about-albert-roberta-and-distilbert-11a74334b2da/

##### Overview of combination
DistilBERT can be tuned for a sequence classification task where the model directly predicts the resolver group based on the text input. Essentially text input, tokenizer, DistilBERT, classification head = predicted resolver group. The model outputs a probabilistic output similar to candidate #1, allowing us to select the highest probability as the class prediction.

Pros:
- Captures relationship between words to better understand context/ meaning
- Able to better understand variations in wording for the same incident type
- Often performs better on text classification tasks vs basic classifiers

Cons:
- Significantly more computationally expensive than traditional NLP models
- Requires additional dependencies like transformers and other libraries
- Harder to interpret compared with a linear model due to the complexity of the neural network

Summary:
DistilBERT will provide us with a more advanced approach to text classification by capturing relationships between words instead of just the individual words alone. This should in theory mean improved classification performance when the words used in descriptions vary. However, it will add complexity, longer training times and additional requirements for libraries/ infrastructure which can all be negatives.

____________________________________________________________________________________________________________________________________________

## Candidate #3
### SetFit - sentence transformer + classifier
##### Sentence transformer
Sentence transformers convert text into vector representations aka embeddings. These embeddings represent the meaning of a sentence rather than individual words, similar to candidate #2. This enables sentences with similar meanings to produce similar vector repsresentations so a model can make that link and use it to help classify a record.

References: 
- https://huggingface.co/docs/setfit/index
- https://sbert.net/index.html

##### SetFit
SetFit is a framework that combines sentence transformers with a classifier. Instead of full tuning a transformer model, setfit generates embeddings using a pre-trained transformer model and then trains a smaller classifier on top of those embeddings. This way it can reduce training time whilst still getting the benefit of the context and understanding provided by transformer models.

References: 
- https://huggingface.co/docs/setfit/index
- https://medium.com/@youssefchamrah/setfit-unpacked-when-sentence-transformers-go-to-gym-for-classification-muscle-56c16d9e69de

##### Overview of combination
SetFit looks simple when looking at it process-wise due to it's pre-trained nature. text input, sentence embeddings, classifier = predicted resolver group. It is also significantly faster than tuning a transformer model, again due to it's pre-trained nature.

Pros:
- Captures context and meaning using transformer embeddings
- Faster training compared with a full transformer tuning
- Lower computational cost than models like candidate #2

Cons:
- Introduces another modelling framework and dependency to the solution
- Less common to find implementations/ examples and research than the other candidates
- If there is lots of labelled training data, the benefit of the pre-trained nature is lowered significantly

Summary:
SetFit fits somewhere betweeen candidate #1 a traditional ML approach and candidate #2 a transformer based model. It can provide us with context and meaning similar to the transformer models whilst keeping training time lower due to it being pre-trained. However, if there is lots of training data which is labelled with the target we are trying to predict, it's benefit can be lessened significantly.

____________________________________________________________________________________________________________________________________________

# Decision Matrix
##### Candidates:
- Candidate 1: TF-IDF + Logsitic regression
- Candidate 2: DistilBERT
- Candidate 3: SetFit

##### Criteria:
- `Expected performance`: Likelihood of the model performing well on multi-class text classification
- `Ability to capture context and meaning` from text inputs: Our text descriptions are free text and will vary so meaning/ context will likely be very important
- `Explainability`: Need to be able to explain model behaviour to stakeholders
- `Complexity`: Trade off between complexity and performance with lower complexity preferred
- `Training and inference cost`: Keep cost and overhead low
- `Suitability for confidence based routing`: Must support probabalistic outputs so we can route tickets for manual review if confidence is low

##### Matrix:
| Criteria                                      | TF-IDF + Logistic regression | DistilBERT | SetFit      |
| ----------------------------------------------|----------------------------- | ---------- | ----------- |
| Expected performance                          | Medium                       | High       | Medium/High |
| Ability to capture context and meaning        | Low                          | High       | Medium/High |
| Explainability                                | High                         | Low        | Medium      |
| Complexity                                    | Low                          | High       | Medium      |
| Training and inference cost                   | Low                          | High       | Medium      |
| Suitability for confidence based routing      | High                         | High       | Medium/High |
| Fit for this project                          | High                         | High       | Medium      |

##### Decision:
1. TF-IDF + Logistic regression: Implement as a baseline model. This will provide us with a fast, simple and explainable baseline to reference and based on my research is a standard approach for classic text classification.
2. DistilBERT: Implement as the main challenger model vs our baseline model. It is a transformer model based on the powerful BERT model which based on research should yeild very strong performance.
3. SetFit: Not to be implemented for now. Although it may be an efficient way of approaching the problem when labelled data is limited, we have a significant amount of labelled data available for training so it makes less sense than tuning our own transformer based model on our specific data. 

Candidate 1 and 2 will be implemented for evaluation and comparison.

____________________________________________________________________________________________________________________________________________

# Load data and artefacts

In [1]:
# Imports
from pathlib import Path
from datetime import datetime
import io
import json
import yaml
import pandas as pd
from minio import Minio

In [2]:
# Load config from yaml
CONFIG_FILE = Path("../config/config.yaml")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Missing config file: {CONFIG_FILE.resolve()}")

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("Config loaded")

Config loaded


In [3]:
# Read .env for credentials
def load_env_file(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing env file: {path.resolve()}")

    env = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip()
    return env

# Env file location in repo
ENV_FILE = Path(f"{cfg['storage']['minio']['env_file']}")
# Storage endpoint and config
MINIO_ENDPOINT = cfg["storage"]["minio"]["endpoint"]
MINIO_SECURE = cfg["storage"]["minio"]["secure"]

# Authenticate to file storage list buckets to confirm connection
env = load_env_file(ENV_FILE)
client = Minio(
    MINIO_ENDPOINT,
    access_key=env["MINIO_ROOT_USER"],
    secret_key=env["MINIO_ROOT_PASSWORD"],
    secure=MINIO_SECURE,
)

print("Connected buckets:", [b.name for b in client.list_buckets()])

Connected buckets: ['data-profile-test', 'incident-pipeline', 'incident-pipeline-test', 'mlflow-artifacts']


In [4]:
# Load latest gold train, valid and test data from storage
gold_prefix_root = cfg["storage"]["minio"]["gold_training_prefix_root"].rstrip("/")
test_bucket = "incident-pipeline-test"

# Find latest timestamped gold run
run_tags = sorted({
    obj.object_name[len(gold_prefix_root) + 1:].split("/", 1)[0]
    for obj in client.list_objects(test_bucket, prefix=f"{gold_prefix_root}/", recursive=True)
    if obj.object_name.startswith(f"{gold_prefix_root}/") and len(obj.object_name.split("/")) >= 4
})

# Error if no gold runs found
if not run_tags:
    raise RuntimeError(f"No gold runs found under: {gold_prefix_root}")

# Use the latest gold run for training
latest_gold_run_tag = run_tags[-1]
latest_gold_run_prefix = f"{gold_prefix_root}/{latest_gold_run_tag}"

# Save this for model artifact traceability
training_data_run_tag = latest_gold_run_tag

# Paths for train, valid and test files
# Path for label mapping file
train_path = f"{latest_gold_run_prefix}/train.parquet"
valid_path = f"{latest_gold_run_prefix}/valid.parquet"
test_path = f"{latest_gold_run_prefix}/test.parquet"
label_mapping_path = f"{latest_gold_run_prefix}/label_mapping.json"

# Function to load parquet files
def load_parquet_from_storage(bucket: str, object_name: str) -> pd.DataFrame:
    resp = client.get_object(bucket, object_name)
    try:
        return pd.read_parquet(io.BytesIO(resp.read()))
    finally:
        resp.close()
        resp.release_conn()

# Function to load json files
def load_json_from_storage(bucket: str, object_name: str) -> dict:
    resp = client.get_object(bucket, object_name)
    try:
        return json.loads(resp.read().decode("utf-8"))
    finally:
        resp.close()
        resp.release_conn()

# Files to dfs
train_df = load_parquet_from_storage(test_bucket, train_path)
valid_df = load_parquet_from_storage(test_bucket, valid_path)
test_df = load_parquet_from_storage(test_bucket, test_path)
label_mapping = load_json_from_storage(test_bucket, label_mapping_path)

# Print summary info on the run and loaded data
print("Latest gold run tag:", latest_gold_run_tag)
print("Latest gold prefix:", latest_gold_run_prefix)
print("Train rows:", len(train_df))
print("Valid rows:", len(valid_df))
print("Test rows:", len(test_df))
print("Label mapping size:", len(label_mapping["classes"]))


Latest gold run tag: 20260308_205827
Latest gold prefix: gold/training/20260308_205827
Train rows: 7999
Valid rows: 1001
Test rows: 1000
Label mapping size: 12


In [5]:
# Label mapping check
display(label_mapping["classes"])

['App Support - ERP',
 'App Support - Finance',
 'App Support - HRIS',
 'App Support - M365',
 'App Support - Microsoft Fabric',
 'App Support - Power BI',
 'App Support - Power Platform',
 'End User Compute',
 'Identity and User Access',
 'Integration & Middleware',
 'Network Ops',
 'Security Operations']

# Validate data

In [6]:
# validate columns are present in all splits and check for leakage
# required cols
required_cols = ["sys_id", "text", "label_final", "label_id"]

# check for required cols in each split
for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

    # Print now number, null rate, empty rate and number of labels in each split
    print(f"\n{name.upper()} checks")
    print("Rows:", len(df))
    print("Empty text rate:", f"{df['text'].fillna('').astype(str).str.strip().eq('').mean():.2%}")
    print("Null label rate:", f"{df['label_final'].isna().mean():.2%}")
    print("Unique labels:", df["label_final"].nunique())

# Collect sys_ids for leakage check
train_ids = set(train_df["sys_id"])
valid_ids = set(valid_df["sys_id"])
test_ids = set(test_df["sys_id"])

# Print overlap of sys_ids between splits to check for leakage
print("\nSplit leakage")
print("Train/Valid overlap:", len(train_ids & valid_ids))
print("Train/Test overlap :", len(train_ids & test_ids))
print("Valid/Test overlap :", len(valid_ids & test_ids))


TRAIN checks
Rows: 7999
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

VALID checks
Rows: 1001
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

TEST checks
Rows: 1000
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

Split leakage
Train/Valid overlap: 0
Train/Test overlap : 0
Valid/Test overlap : 0


# Implementation candidate #1
TF-IDF + Logistic regression (baseline model)

Starting simple with candidate 1 before we build the candidate 2 transformer model to give us a simple, explanable and computationally cheap reference point. This will help us establish what is possible with the data and help give us a benchmark to better understand the performance of candidate 2.

No heavy tuning to start with as iterations of candidate 1 may be considered for production, but this first model will be our benchmark only.

In [18]:
# Imports for c1 (Candidate #1)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import numpy as np

In [19]:
# Basic baseline config
baseline_pipeline = Pipeline([
    (
    "TFIDF", TfidfVectorizer(
        max_features=50000, # Cap on features to prevent overfitting and reduce noise
        ngram_range=(1, 2), # Allows the model to use both single words and pairs of words for more context capture
        min_df=2 # Filter out rare words to reduce noise
        )
    ),
    (
    "LogisticRegression", LogisticRegression(
        max_iter=1000, # Max iterations
        multi_class="multinomial", # Use multinomial for multi-class classification
        random_state=42
        )
    )
])

# Data for training
X_train = train_df["text"]
y_train = train_df["label_final"]

# Data for validation
X_valid = valid_df["text"]
y_valid = valid_df["label_final"]

# Train the baseline model
baseline_pipeline.fit(X_train, y_train)

# Predictions and probabilities on the validation split
valid_preds = baseline_pipeline.predict(X_valid)
valid_probs = baseline_pipeline.predict_proba(X_valid)

In [20]:
# Baseline model evaluation
baseline_val_accuracy = accuracy_score(y_valid, valid_preds)
baseline_val_macro_f1 = f1_score(y_valid, valid_preds, average="macro")

print(f"BL val acc:", baseline_val_accuracy)
print(f"BL val macro F1", baseline_val_macro_f1)

baseline_classification_report = classification_report(y_valid, valid_preds)
print(baseline_classification_report)


BL val acc: 0.8351648351648352
BL val macro F1 0.8635150830971927
                                precision    recall  f1-score   support

             App Support - ERP       1.00      0.80      0.89        41
         App Support - Finance       0.97      0.82      0.89        38
            App Support - HRIS       1.00      0.88      0.94        34
            App Support - M365       0.58      0.95      0.72       176
App Support - Microsoft Fabric       1.00      0.82      0.90        83
        App Support - Power BI       0.92      0.83      0.87       112
  App Support - Power Platform       0.90      0.83      0.86       103
              End User Compute       0.96      0.79      0.87        98
      Identity and User Access       0.98      0.72      0.83        87
      Integration & Middleware       1.00      0.73      0.84        33
                   Network Ops       0.84      0.85      0.85       134
           Security Operations       1.00      0.82      0.90        

The model performed reasonably well 
validation accuracy: ~0.835
validation macro f1: ~0.864

This represends our baseline and the model to compare all others and tuning against.

An accuracy of ~83% means the model is correctly predicting the resolver group for the majority of incidents. A macro f1 score of ~0.86 also tells us that performance is well balanced acorss different resolver groups, with no one class dominating performance/ skewing the results.

The model appears to perform strongly on classes such as:
- App Support - HRIS
- App Support - Microsoft Fabric
- App Support - ERP
- Security Operations

All of these classes are in and around .9-.94 for F1 scores, which tells us the descriptions for these systems often contain distinctive wording that TF-IDF is able to capture.

The model does perform poorly on some classes however:
- App Support - M365
- Network Ops
- Identity and User Access
- Integration & Middleware

These classes have lower pecision or recall, telling us the model is sometimes confusing them with other classes. This is expected with TF-IDF since the model is only looking at singular or short word combos rather than understanding the meaning of a whole sentence. 

In [21]:
baseline_max_probs = valid_probs.max(axis=1)
baseline_conf_threshold = 0.70

baseline_manual_review_mask = baseline_max_probs < baseline_conf_threshold
baseline_manual_review_rate = baseline_manual_review_mask.mean()

print("Average max probability:", round(float(baseline_max_probs.mean()), 4))
print(f"Manual review rate at threshold {baseline_conf_threshold}:", round(float(baseline_manual_review_rate), 4))
print("Lowest confidence:", round(float(baseline_max_probs.min()), 4))
print("Highest confidence:", round(float(baseline_max_probs.max()), 4))

Average max probability: 0.7245
Manual review rate at threshold 0.7: 0.2837
Lowest confidence: 0.1278
Highest confidence: 0.9661


Since the requirements of this project require a human in the loop element for low confidence incidents, we need to take a look at the models prediction confidence.

The average confidence is ~0.72
The amount under the threshold is 28% so this would be the number ending up for manual review
The lowest confidence prediction we had was 0.13
And the highest was 0.97

The confidence threshold was set at 0.70 meaning 28% of predictions fell below this threshold and would have been routed for manual review. This tells us the model is quite confident in a lot of it's predictions but for the ones it struggles with, confidence is very low.

In [22]:
baseline_cm = confusion_matrix(y_valid, valid_preds, labels=baseline_pipeline.classes_)
baseline_cm_df = pd.DataFrame(
    baseline_cm,
    index=baseline_pipeline.classes_,
    columns=baseline_pipeline.classes_
)

baseline_cm_df

,App Support - ERP,App Support - Finance,App Support - HRIS,App Support - M365,App Support - Microsoft Fabric,App Support - Power BI,App Support - Power Platform,End User Compute,Identity and User Access,Integration & Middleware,Network Ops,Security Operations
App Support - ERP,33,1,0,7,0,0,0,0,0,0,0,0
App Support - Finance,0,31,0,6,0,0,0,0,0,0,1,0
App Support - HRIS,0,0,30,3,0,1,0,0,0,0,0,0
App Support - M365,0,0,0,167,0,2,3,0,0,0,4,0
App Support - Microsoft Fabric,0,0,0,9,68,0,0,1,0,0,5,0
App Support - Power BI,0,0,0,15,0,93,0,0,0,0,4,0
App Support - Power Platform,0,0,0,15,0,0,85,1,0,0,2,0
End User Compute,0,0,0,15,0,1,3,77,0,0,2,0
Identity and User Access,0,0,0,16,0,3,2,0,63,0,3,0
Integration & Middleware,0,0,0,9,0,0,0,0,0,24,0,0


From the confusion matrix we can see a few patterns that tells us a bit more about where the model is making mistakes.

1. A lot of classification mistakes end up in M365. This is expected due to the wide ranging set of technologies that make up M365, meaning there is often overlap with other classes like login issues, access problems or specific tool bugs/ glitches.

2. There is some overlap between infrastructure and related teams (i.e. Network ops, End user compute and Identity and user access). These teams can deal with similar problems for different purposes or systems so can have some overlap in the words used to describe problems.

3. For more specialised systems like Fabric, HRIS and Sec ops the model predicts them correctly most of the time. This is due to the distinct wording around issues from the specailised nature of each of these systems. No other systems have cross over with this capabilities leading to lots of key words for TF-IDF to work with.


Overall, the model performance is strong and perfoms well on incidents for specialised systems with distinct terminology best. It's main weakness comes from the lack of context and the restriction to single or pairs of words, this is leading to poor performance on classes where there is overlap in terminology or the meaning of words change based on the context they are in (which is missing).

This model could however be improved further through some basic tuning. For example TF-IDF has a number of additional parameters that can be tuned to influence how text is convered into features. This includes adjusting the ngram_range (single words vs single words and short phrases of different lengths), min and max df to control how common or rare words must be before they can be included as features, and the max_features limit to control the size of the voca used by the model.

On the logistic regression side, the regularisation param 'C' can also be tuned to control how strongly the model penalises more complex decision boundaries. Changing this can improve performance when feature space is very large which can happened often with TF-IDF.

___________________________________________________________________________________________________________________________________________

# Implementation candidate #2
DistilBERT challenger model

Candidate 1 saw good performance overall, especially on specialised classes that had consistent distinct wording. It did however struggle on broader classes that have overlap with one another in terms of the wording or terminology often used to describe problems. 

Candidate 2 should provide better performance on classes with overlaps as it utilises a transformer based model which is designed to handle these types of challenges. Unlike TF-IDF which mainly looks at word frequency and short combinations, DistilBERT is able to look at relationships between words in context. In theory, this should translate to a better understanding that two incidents can mean the same thing even if they are worded differently by the caller.

DistilBERT was the first choice due to it's smaller and faster nature when comapred with the full version of BERT, this should give us a good balance between model capability and computational cost.

At this stage, the model will be un-tuned, taking default configuration from hugging face before we iterate with optimisation. The goal is to establish whether the transformer model itself can provide a meaningful improvement over the baseline before spending time tuning it further.

In [7]:
# Imports for c2 (Candidate #2)
import numpy as np
import pandas as pd
import torch
from datetime import timezone

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)

Dataset + Trainer to follow default/ suggested config from hugging face: https://huggingface.co/docs/transformers/tasks/sequence_classification

Conversion from pandas dataframes to hugging face datasets as the optimal format for the hugging face trainer. 

Everything kept simple and in line with the default configuration provided by hugging face at this stage for initial model.

In [8]:
# Make a copy of the dataframes for train, val and test
c2_train = train_df[["sys_id", "text", "label_id", "label_final"]].copy()
c2_valid = valid_df[["sys_id", "text", "label_id", "label_final"]].copy()
c2_test = test_df[["sys_id", "text", "label_id", "label_final"]].copy()

# Rename target column to 'labels' as is expected by the trainer
c2_train = c2_train.rename(columns={"label_id": "labels"})
c2_valid = c2_valid.rename(columns={"label_id": "labels"})
c2_test = c2_test.rename(columns={"label_id": "labels"})

# Convert from pandas dataframe to dataset
c2_hf_train = Dataset.from_pandas(c2_train, preserve_index=False)
c2_hf_valid = Dataset.from_pandas(c2_valid, preserve_index=False)
c2_hf_test = Dataset.from_pandas(c2_test, preserve_index=False)

# Print rows counts and column names to ensure data shape remains the same
print("HF train rows:", len(c2_hf_train))
print("HF valid rows:", len(c2_hf_valid))
print("HF test rows :", len(c2_hf_test))
print("HF train columns:", c2_hf_train.column_names)

HF train rows: 7999
HF valid rows: 1001
HF test rows : 1000
HF train columns: ['sys_id', 'text', 'labels', 'label_final']


First transformer to try is distilbert/distilbert-base-uncased

Hugging face use this model configuration in their base level tutorial for DistilBERT classification problem, so is a well researched and supproted entry point for the model.
https://huggingface.co/docs/transformers/tasks/sequence_classification

In [9]:
# label_mapping from text value to id acording to the definitions provided by the gold transformation layer
label_to_id = {k: int(v) for k, v in label_mapping["label_to_id"].items()}
id_to_label = {int(k): v for k, v in label_mapping["id_to_label"].items()}

checkpoint = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_mapping["classes"]),
    id2label=id_to_label,
    label2id=label_to_id,
)

print("Checkpoint:", checkpoint)
print("Number of labels:", len(label_mapping["classes"]))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checkpoint: distilbert/distilbert-base-uncased
Number of labels: 12


Tokenisation:
- Maintaining close replication of tutorial as our default config
- No additional hyperparameters included:
    - Pretrained tokeniser: Simple and less time to iterate and evaluate

In [10]:
# Tokenisation function
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized_train = c2_hf_train.map(tokenize_batch, batched=True)
tokenized_valid = c2_hf_valid.map(tokenize_batch, batched=True)
tokenized_test = c2_hf_test.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenized train columns:", tokenized_train.column_names)
print("Example input length:", len(tokenized_train[0]["input_ids"]))

Map:   0%|          | 0/7999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenized train columns: ['sys_id', 'text', 'labels', 'label_final', 'input_ids', 'token_type_ids', 'attention_mask']
Example input length: 134


In [11]:
# Performance metrics function, accuracy_score and macro-f1
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

Initial settings:

In line with hugging face example:
- learning rate = 2e-5
- train batch size = 16
- eval batch size = 16
- epochs = 2
- weight decay = 0.01
- evaluation and saving at each epoch

Project specific choices:
- seed = 42: for repeatability
- laod_best_model_at_end = True: retain the best validation checkpoint
- metric_for_best_model = eval_macro_f1: set to macro f1 as it gives us a better performance guage vs accuracy alone for multi class problems
- report_to = none: Implement, evaluate, iterate - no need to wire up to MLFlow or log until we are happy with our model

In [12]:
# Output path for distilbert
distilbert_output_dir = f"./outputs/distilbert_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

# Training arguments
training_args = TrainingArguments(
    output_dir=distilbert_output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    seed=42,
    report_to="none",
)

print("Training output dir:", distilbert_output_dir)
print(training_args)

Training output dir: ./outputs/distilbert_20260313_110841
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_st

Using trainer as it is the hugging face standard training API for supervised transformer tasks and it gives a lot of functionality without having to write complex code:
1. Training loop management
2. Evaluation each epoch
3. Checkpoint saving
4. Best model loading
5. Consistent metrics

The alternatvie is to write a full pytorch training loop, which would provie more control but at this stage is not required as our base transformer model. 

Train and evaluate this model and then consider performance vs potential tuning or approach tweaks to further enhance performance.

In [14]:
distilbert_trainer = Trainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

distilbert_train_result = distilbert_trainer.train()

print("Training complete")
print("Best checkpoint:", distilbert_trainer.state.best_model_checkpoint)

c:\Users\JHoll\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.958083,0.479136,0.839161,0.873421
2,0.495709,0.462833,0.839161,0.874454


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\JHoll\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training complete
Best checkpoint: ./outputs/distilbert_20260313_110841\checkpoint-1000


Run took a considerable amount of time to train. Retain the full run as a long running baseline, but re-train wth better config allowing for fast iteration before we make a final decision.

Suggested changes:
- Only train over 1 epoch
- Limit max steps to 200 rather than 1000 for prototyping
- Remove repeated evaluation steps
- Remove checkpoint save
- Set tokenisation max length to 128 for fair comparisons


Things that must remain the same for a fair comaprison:
- Dataset and split
- Sequence length (max_length)
- Max steps
- Epochs
- Batch size
- Model/ checkpoint
- Random seed
- Evalation method and metrics

# C2v1 Same model but with repeatable config

In [17]:
training_args_c2v1 = TrainingArguments(
    output_dir=distilbert_output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    max_steps=200,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    load_best_model_at_end=False,
    save_total_limit=1,
    seed=42,
    warmup_steps=0,
    logging_steps=20,
    report_to="none",
)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)


In [18]:
distilbert_trainer_c2v1 = Trainer(
    model=distilbert_model,
    args=training_args_c2v1,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

distilbert_train_result = distilbert_trainer_c2v1.train()

print("Training complete")
print("Global steps:", distilbert_trainer_c2v1.state.global_step)

Step,Training Loss
20,0.482381
40,0.523885
60,0.444571
80,0.527245
100,0.411963
120,0.502657
140,0.459813
160,0.510758
180,0.574477
200,0.397193


Training complete
Global steps: 200


Evaluation on validation split only the same as baseline.

Holding test until we are set on config tuning and final production candidate is select. Then we can use it on selected models to compare directly on a different set to val.

In [20]:
distilbert_valid_output_c2v1 = distilbert_trainer_c2v1.predict(tokenized_valid)

distilbert_valid_logits_c2v1 = distilbert_valid_output_c2v1.predictions
distilbert_valid_labels_c2v1 = distilbert_valid_output_c2v1.label_ids
distilbert_valid_preds_c2v1 = np.argmax(distilbert_valid_logits_c2v1, axis=-1)

distilbert_valid_accuracy_c2v1 = accuracy_score(distilbert_valid_labels_c2v1, distilbert_valid_preds_c2v1)
distilbert_valid_macro_f1_c2v1 = f1_score(distilbert_valid_labels_c2v1, distilbert_valid_preds_c2v1, average="macro")

print("DistilBERT validation accuracy:", round(distilbert_valid_accuracy_c2v1, 4))
print("DistilBERT validation macro F1:", round(distilbert_valid_macro_f1_c2v1, 4))

c:\Users\JHoll\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


DistilBERT validation accuracy: 0.8362
DistilBERT validation macro F1: 0.869


headline metrics ```markdown The DistilBERT model performed reasonably well on the validation set. - **Validation accuracy:** ~0.836 - **Validation macro F1:** ~0.869 This is only a very small improvement over the baseline. Accuracy has barely changed at all and macro F1 has increased slightly, which suggests the transformer model is handling some classes a bit better overall, but not by a major amount. That is quite an important finding because it tells us that the baseline TF-IDF + Logistic Regression model was already quite strong. DistilBERT has not come in and massively outperformed it on headline metrics alone. At this stage the result is best described as **slightly better overall**, rather than clearly superior. ```

The DistilBERT model performed well on the validation set.
Val acc: 0.8362
Val macro f1: 0.869

This is only a marginal improvement over the baseline. Accuracy has not really changed at all and macro f1 has improved very slightly, which suggests the transformer model is handling some classes better overall, but not by a large amount.

In [21]:
label_names_in_order = [id_to_label[i] for i in sorted(id_to_label.keys())]

distilbert_classification_report_c2v1 = classification_report(
    distilbert_valid_labels_c2v1,
    distilbert_valid_preds_c2v1,
    target_names=label_names_in_order,
    digits=2,
)

print(distilbert_classification_report_c2v1)

                                precision    recall  f1-score   support

             App Support - ERP       1.00      0.83      0.91        41
         App Support - Finance       1.00      0.82      0.90        38
            App Support - HRIS       1.00      0.88      0.94        34
            App Support - M365       0.55      0.98      0.70       176
App Support - Microsoft Fabric       1.00      0.82      0.90        83
        App Support - Power BI       0.87      0.81      0.84       112
  App Support - Power Platform       1.00      0.83      0.90       103
              End User Compute       1.00      0.79      0.88        98
      Identity and User Access       1.00      0.71      0.83        87
      Integration & Middleware       1.00      0.73      0.84        33
                   Network Ops       0.94      0.83      0.88       134
           Security Operations       1.00      0.82      0.90        62

                      accuracy                           0.84 

Narrowing in at a class level, the model performs well on a lot of the same specialised classes as the baseline. It still does well on:
- App Support - HRIS
- App Support - ERP
- App Support - Microsoft Fabric
- Security Operations

These are all around the 0.90 or above for F1 score, which tells us that specialised systems with distinct wording are still the easiest classes for the model to identify. There are also some areas where the model looks a bit better than baseline:
- Network Ops
- App Support - Power Platform
- End User Compute

This suggests the transformer model may perform better where there is more wording variation or a bit more context is needed. However, the improvement is not consistent across each class. 

Some classes are still weak or even a bit worse:
- App Support - M365
- Power BI
- Identity and User Access
- Integration & Middleware

All of these classes are still weaker classes so overall this model has not provided a step change in performance output. It improves some classes, but it is still not great in all of the same places we had issues with the baseline model.

In [23]:
distilbert_valid_probs_c2v1 = torch.softmax(torch.tensor(distilbert_valid_logits_c2v1), dim=1).numpy()
distilbert_max_probs_c2v1 = distilbert_valid_probs_c2v1.max(axis=1)

distilbert_conf_threshold = 0.70
distilbert_manual_review_mask_c2v1 = distilbert_max_probs_c2v1 < distilbert_conf_threshold
distilbert_manual_review_rate_c2v1 = distilbert_manual_review_mask_c2v1.mean()

print("Average max probability:", round(float(distilbert_max_probs_c2v1.mean()), 4))
print(f"Manual review rate at threshold {distilbert_conf_threshold}:", round(float(distilbert_manual_review_rate_c2v1), 4))
print("Lowest confidence:", round(float(distilbert_max_probs_c2v1.min()), 4))
print("Highest confidence:", round(float(distilbert_max_probs_c2v1.max()), 4))

Average max probability: 0.8477
Manual review rate at threshold 0.7: 0.1888
Lowest confidence: 0.1419
Highest confidence: 0.997


When looking at our probability outputs with routing for manual review to occur for anything below our set threshold the model performed...:
- Average max probability: 0.8477
- Manual review rate at threshold 0.7: 0.1888
- Lowest confidence: 0.1419
- Highest confidence: 0.997

The model was a lot more confident in it's predictions than the baseline model, resulting in less incidents being routed for manual review with the threshold set at the same 0.70. The baseline had about 28% under the threshold whereas this model has 19% in comparison.

In reality this would result in a huge up tick in performance and business benefit with 9% less tickets requiring manual review.

We do however need to be careful since higher confidence does not always equate to better performance. Confidence improvement is a good thing but not distinctive proor the model is better overall.

In [24]:
distilbert_cm_c2v1 = confusion_matrix(
    distilbert_valid_labels_c2v1,
    distilbert_valid_preds_c2v1,
    labels=list(sorted(id_to_label.keys())),
)

distilbert_cm__c2v1_df = pd.DataFrame(
    distilbert_cm_c2v1,
    index=label_names_in_order,
    columns=label_names_in_order,
)

distilbert_cm__c2v1_df

,App Support - ERP,App Support - Finance,App Support - HRIS,App Support - M365,App Support - Microsoft Fabric,App Support - Power BI,App Support - Power Platform,End User Compute,Identity and User Access,Integration & Middleware,Network Ops,Security Operations
App Support - ERP,34,0,0,6,0,0,0,0,0,0,1,0
App Support - Finance,0,31,0,6,0,0,0,0,0,0,1,0
App Support - HRIS,0,0,30,1,0,3,0,0,0,0,0,0
App Support - M365,0,0,0,173,0,3,0,0,0,0,0,0
App Support - Microsoft Fabric,0,0,0,12,68,2,0,0,0,0,1,0
App Support - Power BI,0,0,0,20,0,91,0,0,0,0,1,0
App Support - Power Platform,0,0,0,17,0,0,85,0,0,0,1,0
End User Compute,0,0,0,19,0,2,0,77,0,0,0,0
Identity and User Access,0,0,0,23,0,1,0,0,62,0,1,0
Integration & Middleware,0,0,0,8,0,1,0,0,0,24,0,0


From the confusion matrix we can see a few patterns that give us an idea of where the model is making mistakes. 

A lot of errors end up in M365 the same as the baseline model. This is till the clearest pattern with a large number of misclassifications being predicted as 'App Support -M365' which tells us this class is till acting like a catch all when the model is unsure. 

This does make sense however due to the wide range of tools M365 covers, so the wording used in these incidents is likely to overlap with other classes. 

Another pattern is overlap between other general operational teams like Network Ops, End User Computer and Identity and User Access.

Overall the confusion matrixs tells us the model has not solved the class overlap issue but it does look a bit stronger in some of the harder areas.

Overall the model has performed marginally better than the baseline but not by much. The headline accuracy and f1 metrics are broadly the same with the marginal gain coming from the confidence improvements rather than raw performance. The main weakness is still the same as the baseline model, with M365 acting as a broad class that attracts a considerable number of misclassifications.

__________________________________________________________________________________________________________________________________________

# Candidate 2 improvement (c2_v2)

c2_v1 model showed that a default configured transformer model only slightly improved on accuracy and macro F1, but provided a large improvement in confidence in predictions. This means the model is already showing some benefit vs the baseline, but not enough yet to clearly justify the additional complexity.

Our evaluation showed us the main weakness is still the broad overlapping classes with M365 and some operational teams accouting for most. This suggests the model is struggling to seperate classes where wording is similar and context matters to derive the meaning.

#### Areas of focus:
1. Preserve more useful text structure: In the first version I combined both short and normal description fields into a single input. For this next phase I will keep them seperate, a lot of valuable information is usually in the short description field with additional context provided by the longer field. 

2. Train the model for longer: Increase the number of epochs to give the model more change to learn patterns. At the momement we may be too restrictive for the model type.

3. Bring in early stopping: This will allow us to expand the training window whilst stopping if the validation performance does not improve.

4. Keep an eye on class weighted loss if the broad class bias continues: No action for now but one to watch in the outputs as we may need to consider targeting action to these classes.

This should give us:
- Better context
- Longer but controlled training window to allow the model to learn
- Watch class-weighted loss and target improvements